# 1. Introduction

This project prepares a clean, aggregated dataset for a Tableau sales dashboard built on the [Online Retail II](https://www.kaggle.com/datasets/lakshmi25npathi/online-retail-dataset) dataset.

The notebook is the **data layer** of the project — its only job is to take raw transactional data, clean it, and export a small set of CSV files in shapes that Tableau can connect to directly. The dashboard itself (`sales_dashboard.twb`) lives in the same folder and consumes these CSVs.

# 2. Business Problem

The previous project (`project-2-automated-sales-report`) produces a static monthly Excel report. That format works for a single sender → recipient hand-off, but it does not scale for an audience that wants to **explore** the data: filter by period, drill into products, compare segments.

This project closes that gap by delivering an interactive Tableau dashboard with:

- Headline KPIs (Revenue, Orders, AOV, Customers) with period-over-period comparison
- Revenue and orders trends over time
- Top products by revenue
- Customer segmentation (ABC) and repeat-vs-one-time mix
- A single date-range selector that drives every chart

# 3. Dataset Description

Same source as projects 1 and 2: the **Online Retail II** dataset — a real UK-based online retailer's transaction log from December 2009 to December 2011, primarily gift-ware sales with a meaningful wholesale share.

Source: https://www.kaggle.com/datasets/lakshmi25npathi/online-retail-dataset

Place `online_retail_II.xlsx` in `data/`, or configure Kaggle API credentials and the notebook will download it automatically.

# 4. Data Preparation

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from kaggle.api.kaggle_api_extended import KaggleApi



In [3]:
Path('data').mkdir(exist_ok=True)

file_path = Path('data/online_retail_II.xlsx')

if not file_path.exists():
    if KaggleApi is None:
        raise RuntimeError(
            'Dataset not found. Either place online_retail_II.xlsx in data/, '
            'or install the kaggle package and configure credentials.'
        )
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files(
        'lakshmi25npathi/online-retail-dataset',
        path='data',
        unzip=True,
    )

df_2009_2010 = pd.read_excel(file_path, sheet_name='Year 2009-2010')
df_2010_2011 = pd.read_excel(file_path, sheet_name='Year 2010-2011')

df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
print(f'Loaded {df.shape[0]:,} rows.')


Loaded 1,067,371 rows.


### 4.1 Data Cleaning

Same cleaning rules as project 1 and project 2, kept in sync so all three projects operate on identical analytical data:

- Standardize column names to lowercase snake_case
- Cast `customer_id` to a nullable integer
- Drop exact duplicate rows
- Drop returns, manual adjustments, and non-product stock codes
- Drop rows with non-positive quantity or price

In [4]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df.rename(columns={'stockcode': 'stock_code', 'invoicedate': 'invoice_date'})

df['customer_id'] = df['customer_id'].astype('Int64')
df = df.drop_duplicates()

df_sales = df[
    (df['quantity'] > 0)
    & (df['price'] > 0)
    & (~df['invoice'].astype(str).str.contains(r'[A-Za-z]', na=False))
    & (~df['stock_code'].astype(str).str.fullmatch(r'[A-Za-z]+', na=False))
].copy()

df_customers = df_sales[df_sales['customer_id'].notna()].copy()

print(f'df_sales:     {df_sales.shape[0]:>8,} rows')
print(f'df_customers: {df_customers.shape[0]:>8,} rows')


df_sales:     1,003,687 rows
df_customers:  776,871 rows


### 4.2 Feature Engineering

Two derived columns drive every downstream metric:

- `revenue` = `quantity × price`
- `invoice_month` — calendar month of the invoice, used for monthly aggregation

In [5]:
df_sales['invoice_month'] = df_sales['invoice_date'].dt.to_period('M')
df_sales['revenue']       = df_sales['quantity'] * df_sales['price']
df_customers['invoice_month'] = df_customers['invoice_date'].dt.to_period('M')
df_customers['revenue']       = df_customers['quantity'] * df_customers['price']


# 5. Product Labels

The raw `description` field is in ALL CAPS (e.g. `PAPER CHAIN KIT 50'S CHRISTMAS`). ALL CAPS is fine in a transactional database, but it is hostile in a dashboard — every product title is shouting and the eye has nothing to anchor on.

We normalize descriptions to **Title Case**, with one twist: a naive `str.title()` capitalizes the letter after every apostrophe, turning `50's` into `50'S`. We instead split each description by whitespace and apply `str.capitalize()` per word, which preserves possessive `'s` correctly.

We also keep a `stock_code → description` mapping (the most frequent description per stock code), which removes the long tail of typos and minor variants that exist in the raw data.

In [6]:
def title_case(text: str) -> str:
    """Title-case a string while preserving possessives like 50's."""
    if not isinstance(text, str):
        return text
    return ' '.join(word.capitalize() for word in text.split())

product_labels = (
    df_sales.loc[
        df_sales['stock_code'].notna() & df_sales['description'].notna(),
        ['stock_code', 'description'],
    ]
    .assign(
        stock_code=lambda d: d['stock_code'].astype(str).str.strip(),
        description=lambda d: d['description'].astype(str).str.strip().map(title_case),
    )
    .groupby(['stock_code', 'description'], as_index=False)
    .size()
    .sort_values(['stock_code', 'size', 'description'], ascending=[True, False, True])
    .drop_duplicates(subset='stock_code')
    .rename(columns={'description': 'product_description'})
    .drop(columns='size')
)

product_labels.head(10)


,stock_code,product_description
0,10002,Inflatable Political Globe
1,10002R,Robot Pencil Sharpner
2,10080,Groovy Cactus Inflatable
3,10109,Bendy Colour Pencils
4,10120,Doggy Rubber
5,10123C,Hearts Wrapping Tape
6,10123G,Army Camo Wrapping Tape
7,10124A,Spots On Red Bookcover Tape
8,10124G,Army Camo Bookcover Tape
9,10125,Mini Funky Design Tapes


# 6. Aggregations for Tableau

Five aggregates back the dashboard. Each is exported as a CSV in section 7.

| Output | Granularity | Drives |
|---|---|---|
| `monthly_trends.csv` | one row per day | KPI cards, Revenue trend, Orders trend |
| `top_products.csv` | one row per (day × product) | Top products by revenue chart |
| `abc_product_summary.csv` | one row per ABC segment | Revenue by segment chart |
| `abc_customer_summary.csv` | one row per ABC segment | (optional) customer ABC view |
| `repeat_vs_onetime.csv` | one row per customer type | Repeat vs one-time chart |

The two trend-related files (`monthly_trends`, `top_products`) are kept at **daily** granularity so the dashboard's `Date Selector` parameter can resolve any window — last 30 days, current month, last year, etc. Tableau aggregates back up to the right level.

### 6.1 ABC Analysis (Products)

In [7]:
abc_products = (
    df_sales.assign(stock_code=lambda d: d['stock_code'].astype(str).str.strip())
            .groupby('stock_code', as_index=False)
            .agg(revenue=('revenue', 'sum'))
            .sort_values('revenue', ascending=False)
)
abc_products['revenue_share']    = abc_products['revenue'] / abc_products['revenue'].sum()
abc_products['cumulative_share'] = abc_products['revenue_share'].cumsum()
abc_products['abc_segment'] = np.select(
    [abc_products['cumulative_share'] <= 0.80,
     abc_products['cumulative_share'] <= 0.95],
    ['A', 'B'],
    default='C',
)
abc_products.head()


,stock_code,revenue,revenue_share,cumulative_share,abc_segment
1638,22423,330590.32,0.016815,0.016815,A
4300,85123A,257724.71,0.013109,0.029925,A
4272,85099B,180569.34,0.009185,0.039109,A
2798,23843,168469.60,0.008569,0.047678,A
3147,47566,148318.28,0.007544,0.055222,A


### 6.2 ABC Analysis (Customers)

In [8]:
abc_customers = (
    df_customers.groupby('customer_id', as_index=False)
                .agg(revenue=('revenue', 'sum'))
                .sort_values('revenue', ascending=False)
)
abc_customers['revenue_share']    = abc_customers['revenue'] / abc_customers['revenue'].sum()
abc_customers['cumulative_share'] = abc_customers['revenue_share'].cumsum()
abc_customers['abc_segment'] = np.select(
    [abc_customers['cumulative_share'] <= 0.80,
     abc_customers['cumulative_share'] <= 0.95],
    ['A', 'B'],
    default='C',
)
abc_customers.head()


,customer_id,revenue,revenue_share,cumulative_share,abc_segment
5667,18102,580987.04,0.034011,0.034011,A
2263,14646,526751.52,0.030836,0.064846,A
1779,14156,304719.88,0.017838,0.082684,A
2521,14911,279492.79,0.016361,0.099046,A
5027,17450,244784.25,0.014330,0.113375,A


### 6.3 Repeat vs One-Time Customers

In [9]:
customer_order_counts = (
    df_sales.groupby('customer_id')['invoice']
            .nunique()
            .reset_index(name='order_count')
)
customer_order_counts['customer_type'] = np.where(
    customer_order_counts['order_count'] > 1, 'Repeat', 'One-Time'
)

df_customer_type = df_sales.merge(
    customer_order_counts[['customer_id', 'customer_type']],
    on='customer_id', how='left'
)

repeat_vs_onetime = (
    df_customer_type.groupby('customer_type')
    .agg(
        customers=('customer_id', 'nunique'),
        orders=('invoice', 'nunique'),
        revenue=('revenue', 'sum'),
    )
    .reset_index()
)
for col, total_col in [('customers', 'customers'), ('orders', 'orders'), ('revenue', 'revenue')]:
    repeat_vs_onetime[f'{col}_share_pct'] = (
        repeat_vs_onetime[col] / repeat_vs_onetime[total_col].sum() * 100
    ).round(2)
repeat_vs_onetime['avg_revenue_per_customer'] = (repeat_vs_onetime['revenue'] / repeat_vs_onetime['customers']).round(2)
repeat_vs_onetime['avg_orders_per_customer']  = (repeat_vs_onetime['orders']  / repeat_vs_onetime['customers']).round(2)
repeat_vs_onetime


,customer_type,customers,orders,revenue,customers_share_pct,orders_share_pct,revenue_share_pct,avg_revenue_per_customer,avg_orders_per_customer
0,One-Time,1619,1619,5.567250e+05,27.66,4.42,3.26,343.87,1.00
1,Repeat,4234,35024,1.652581e+07,72.34,95.58,96.74,3903.12,8.27


# 7. Tableau Export

Five CSVs written to `tableau_data/`. The `.twb` workbook in this folder connects to these files by relative path, so as long as the folder structure is preserved the dashboard opens without re-mapping data sources.

In [10]:
tableau_path = Path('tableau_data')
tableau_path.mkdir(exist_ok=True)


### 7.1 monthly_trends.csv

Daily aggregation of revenue, orders, AOV, and unique customers. Tableau rolls these up to whatever granularity the active date filter implies.

In [11]:
monthly_trends_tableau = (
    df_sales
    .assign(month=pd.to_datetime(df_sales['invoice_date']).dt.date)
    .groupby('month', as_index=False)
    .agg(
        revenue=('revenue', 'sum'),
        orders=('invoice', 'nunique'),
        unique_customers=('customer_id', 'nunique'),
    )
    .assign(aov=lambda d: (d['revenue'] / d['orders']).round(2))
    .rename(columns={
        'month':            'Month',
        'revenue':          'Revenue',
        'orders':           'Orders',
        'unique_customers': 'Unique_Customers',
        'aov':              'AOV',
    })
    [['Month', 'Revenue', 'Orders', 'AOV', 'Unique_Customers']]
)
monthly_trends_tableau.to_csv(tableau_path / 'monthly_trends.csv', index=False)
monthly_trends_tableau.head()


,Month,Revenue,Orders,AOV,Unique_Customers
0,2009-12-01,52438.61,117,448.19,89
1,2009-12-02,61873.10,115,538.03,94
2,2009-12-03,73109.78,124,589.60,106
3,2009-12-04,39349.21,88,447.15,76
4,2009-12-05,9803.05,30,326.77,26


### 7.2 top_products.csv

Per-product, per-day revenue and units sold. Description is joined from `product_labels` and is already in clean Title Case from section 5.

In [12]:
top_products_tableau = (
    df_sales
    .assign(
        Month=pd.to_datetime(df_sales['invoice_date']).dt.date,
        Stock_Code=lambda d: d['stock_code'].astype(str).str.strip(),
    )
    .groupby(['Month', 'Stock_Code'], as_index=False)
    .agg(
        Revenue=('revenue', 'sum'),
        Units_Sold=('quantity', 'sum'),
        Orders=('invoice', 'nunique'),
    )
    .merge(
        product_labels.rename(columns={
            'stock_code': 'Stock_Code',
            'product_description': 'Description',
        }),
        on='Stock_Code', how='left',
    )
    .assign(Description=lambda d: d['Description'].fillna('Unknown'))
    [['Month', 'Stock_Code', 'Description', 'Revenue', 'Units_Sold', 'Orders']]
)
top_products_tableau.to_csv(tableau_path / 'top_products.csv', index=False)
top_products_tableau.head()


,Month,Stock_Code,Description,Revenue,Units_Sold,Orders
0,2009-12-01,10002,Inflatable Political Globe,10.2,12,1
1,2009-12-01,10120,Doggy Rubber,12.6,60,1
2,2009-12-01,10123C,Hearts Wrapping Tape,3.9,3,1
3,2009-12-01,10123G,Army Camo Wrapping Tape,3.4,2,1
4,2009-12-01,10125,Mini Funky Design Tapes,5.1,5,2


### 7.3 abc_product_summary.csv

In [13]:
abc_product_summary_tableau = (
    abc_products
    .groupby('abc_segment', as_index=False)
    .agg(products=('stock_code', 'count'), revenue=('revenue', 'sum'))
    .assign(revenue_share_pct=lambda d: (d['revenue'] / d['revenue'].sum() * 100).round(1))
    .rename(columns={
        'abc_segment':       'Segment',
        'products':          'Products',
        'revenue':           'Revenue',
        'revenue_share_pct': 'Revenue_Share_Pct',
    })
    .sort_values('Segment')
)
abc_product_summary_tableau.to_csv(tableau_path / 'abc_product_summary.csv', index=False)
abc_product_summary_tableau


,Segment,Products,Revenue,Revenue_Share_Pct
0,A,1040,1.572579e+07,80.0
1,B,1283,2.950671e+06,15.0
2,C,2581,9.835112e+05,5.0


### 7.4 abc_customer_summary.csv

In [14]:
abc_customer_summary_tableau = (
    abc_customers
    .groupby('abc_segment', as_index=False)
    .agg(customers=('customer_id', 'count'), revenue=('revenue', 'sum'))
    .assign(revenue_share_pct=lambda d: (d['revenue'] / d['revenue'].sum() * 100).round(1))
    .rename(columns={
        'abc_segment':       'Segment',
        'customers':         'Customers',
        'revenue':           'Revenue',
        'revenue_share_pct': 'Revenue_Share_Pct',
    })
    .sort_values('Segment')
)
abc_customer_summary_tableau.to_csv(tableau_path / 'abc_customer_summary.csv', index=False)
abc_customer_summary_tableau


,Segment,Customers,Revenue,Revenue_Share_Pct
0,A,1352,1.366540e+07,80.0
1,B,1895,2.562793e+06,15.0
2,C,2606,8.543394e+05,5.0


### 7.5 repeat_vs_onetime.csv

In [15]:
repeat_vs_onetime_tableau = (
    repeat_vs_onetime
    .rename(columns={
        'customer_type':            'Customer_Type',
        'customers':                'Customers',
        'orders':                   'Orders',
        'revenue':                  'Revenue',
        'revenue_share_pct':        'Revenue_Share_Pct',
        'customers_share_pct':      'Customer_Share_Pct',
        'avg_revenue_per_customer': 'Avg_Revenue_Per_Customer',
        'avg_orders_per_customer':  'Avg_Orders_Per_Customer',
    })
    [['Customer_Type', 'Customers', 'Orders', 'Revenue',
      'Revenue_Share_Pct', 'Customer_Share_Pct',
      'Avg_Revenue_Per_Customer', 'Avg_Orders_Per_Customer']]
)
repeat_vs_onetime_tableau.to_csv(tableau_path / 'repeat_vs_onetime.csv', index=False)
repeat_vs_onetime_tableau


,Customer_Type,Customers,Orders,Revenue,Revenue_Share_Pct,Customer_Share_Pct,Avg_Revenue_Per_Customer,Avg_Orders_Per_Customer
0,One-Time,1619,1619,5.567250e+05,3.26,27.66,343.87,1.00
1,Repeat,4234,35024,1.652581e+07,96.74,72.34,3903.12,8.27


### 7.6 Summary

In [16]:
print('Tableau export complete -> tableau_data/\n')
for f in sorted(tableau_path.iterdir()):
    rows = pd.read_csv(f).shape[0]
    print(f'  {f.name:<35} {rows:>6,} rows')


Tableau export complete -> tableau_data/

  abc_customer_summary.csv                 3 rows
  abc_product_summary.csv                  3 rows
  monthly_trends.csv                     604 rows
  repeat_vs_onetime.csv                    2 rows
  top_products.csv                    531,431 rows


# 8. Dashboard

The Tableau workbook `sales_dashboard.twb` consumes the five CSVs above and assembles the interactive dashboard.

**To open:** `Tableau Desktop` / `Tableau Public Desktop` → `File → Open` → `sales_dashboard.twb`. Tableau will resolve the data sources via relative path to `tableau_data/`.

### Dashboard Components

| Section | Worksheet(s) | Source |
|---|---|---|
| Header | (text object) | — |
| KPI cards | KPI Revenue, KPI Orders, KPI AOV, KPI Customers | `monthly_trends.csv` |
| Revenue trend | Revenue by date | `monthly_trends.csv` |
| Orders trend | Orders by month | `monthly_trends.csv` |
| Top products | Top 10 products | `top_products.csv` |
| Customer mix | One Time vs Repeat Customers | `repeat_vs_onetime.csv` |
| Segment | ABC by products | `abc_product_summary.csv` |

### Interactivity

A single `Date Selector` parameter at the top right drives every chart that has a date dimension — KPIs, Revenue trend, Orders trend, Top products. ABC and customer-mix views are computed over the full period by design (they describe structure, not trend).